# 07 — Model Evaluation & Comparison

Aggregate all model metrics, run Diebold-Mariano tests, and produce thesis-ready charts.

## Test-set summary

| Group | Frequency | Test period | n obs |
|---|---|---|---|
| ARIMA / ARIMAX (weekly) | Weekly | 2023–2024 | ~105 weeks |
| U-MIDAS | Weekly | 2023–2024 | ~105 weeks |
| ARIMA / ARIMAX (daily) | Daily | 2023–2024 | ~500 days |
| LSTM variants | Daily | 2023–2024 | ~480 days |

**Note:** weekly and daily models cannot be directly compared via Diebold-Mariano tests
(different test-set granularity). DM tests are run within each frequency group.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import itertools, os

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')


## 1. Load all metrics

In [ ]:
metrics_map = {
    'metrics_arima.csv':        'Weekly',   # ARIMA / ARIMAX / Naive (weekly)
    'metrics_midas.csv':        'Weekly',   # U-MIDAS (weekly)
    'metrics_arima_daily.csv':  'Daily',    # ARIMA / ARIMAX (daily)
    'metrics_lstm.csv':         'Daily',    # LSTM variants (daily)
}

frames = []
for fname, freq in metrics_map.items():
    fpath = f'../data/processed/{fname}'
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df['frequency'] = freq
        # tag model family
        if 'lstm' in fname:
            df['family'] = 'LSTM'
        elif 'midas' in fname:
            df['family'] = 'MIDAS'
        else:
            df['family'] = 'ARIMA'
        frames.append(df)
        print(f'Loaded {fname}: {len(df)} rows')
    else:
        print(f'NOT FOUND: {fpath}')

all_metrics = pd.concat(frames, ignore_index=True)

# Normalise column name
all_metrics.rename(columns={'dir_acc': 'da'}, inplace=True)
for col in ['rmse', 'mae', 'da', 'wda']:
    if col not in all_metrics.columns:
        all_metrics[col] = np.nan

display(all_metrics[['model', 'frequency', 'family', 'rmse', 'mae', 'da', 'wda']].round(4))


## 2. Weekly model comparison

In [ ]:
weekly = all_metrics[all_metrics['frequency'] == 'Weekly'].copy()
weekly_disp = weekly[['model', 'family', 'rmse', 'mae', 'da', 'wda']].sort_values('da', ascending=False)

(weekly_disp.style
    .highlight_max(subset=['da', 'wda'], color='lightgreen')
    .highlight_min(subset=['rmse', 'mae'], color='lightgreen')
    .format({'rmse': '{:.5f}', 'mae': '{:.5f}', 'da': '{:.3f}', 'wda': '{:.3f}'})
    .set_caption('Weekly models — sorted by DA'))


## 3. Daily model comparison

In [ ]:
daily = all_metrics[all_metrics['frequency'] == 'Daily'].copy()
daily_disp = daily[['model', 'family', 'rmse', 'mae', 'da', 'wda']].sort_values('da', ascending=False)

(daily_disp.style
    .highlight_max(subset=['da', 'wda'], color='lightgreen')
    .highlight_min(subset=['rmse', 'mae'], color='lightgreen')
    .format({'rmse': '{:.5f}', 'mae': '{:.5f}', 'da': '{:.3f}', 'wda': '{:.3f}'})
    .set_caption('Daily models — sorted by DA'))


## 4. Visual comparison — DA and WDA by model

In [ ]:
def plot_da_bars(ax, models_df, metric, title, xlim=(0.35, 0.85)):
    grp = models_df.sort_values(metric).dropna(subset=[metric])
    palette = {'ARIMA': '#2196F3', 'MIDAS': '#4CAF50', 'LSTM': '#FF5722'}
    colors = [palette.get(f, '#9E9E9E') for f in grp['family']]
    bars = ax.barh(grp['model'], grp[metric], color=colors, edgecolor='white')
    ax.axvline(0.5, color='red', ls='--', lw=1.2, label='Random (0.50)')
    ax.set_xlim(*xlim)
    ax.set_xlabel(metric.upper())
    ax.set_title(title, fontweight='bold', fontsize=10)
    for bar, val in zip(bars, grp[metric]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=7)
    # Legend patches
    from matplotlib.patches import Patch
    handles = [Patch(color=c, label=l) for l, c in palette.items()
               if l in grp['family'].values]
    handles.append(plt.Line2D([0], [0], color='red', ls='--', label='Random'))
    ax.legend(handles=handles, fontsize=7, loc='lower right')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

plot_da_bars(axes[0, 0], weekly, 'da',  'Weekly — Directional Accuracy')
plot_da_bars(axes[0, 1], weekly, 'wda', 'Weekly — Weighted DA')
plot_da_bars(axes[1, 0], daily,  'da',  'Daily — Directional Accuracy')
plot_da_bars(axes[1, 1], daily,  'wda', 'Daily — Weighted DA')

plt.suptitle('Model Comparison: Directional Accuracy', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/da_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Diebold-Mariano tests

H₀: equal predictive accuracy (squared error loss).
p < 0.05 → reject H₀ at 5% level.

Baseline for weekly: **ARIMAX rolling (lagged)**
Baseline for daily:  **ARIMAX rolling (lagged)** and **LSTM-EXOG**


In [ ]:
def diebold_mariano(e1, e2, h=1):
    d = e1**2 - e2**2
    T = len(d)
    d_bar = d.mean()
    gamma0 = np.var(d, ddof=1)
    gammas = [np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h)]
    nw_var = gamma0 + 2 * sum(gammas)
    dm_stat = d_bar / np.sqrt(max(nw_var, 1e-12) / T)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {'DM stat': round(dm_stat, 3), 'p-value': round(p_value, 4),
            'Reject H₀ (5%)': p_value < 0.05}


def run_dm_tests(pred_files, baseline_name, label):
    """Load prediction CSVs and run DM tests vs baseline."""
    preds = {}
    for name, fpath in pred_files.items():
        if os.path.exists(fpath):
            df = pd.read_csv(fpath, index_col=0, parse_dates=True)
            preds[name] = df
    if baseline_name not in preds:
        print(f'{label}: baseline {baseline_name!r} not found. Run the corresponding notebook.')
        return
    base = preds[baseline_name]
    rows = []
    for name, df in preds.items():
        if name == baseline_name:
            continue
        merged = base.join(df, rsuffix='_alt', how='inner')
        act = merged['actual'].values
        e1  = act - merged['predicted'].values
        e2  = act - merged['predicted_alt'].values
        res = diebold_mariano(e1, e2)
        rows.append({'Baseline': baseline_name, 'Alternative': name, **res})
    if rows:
        print(f'\n── {label} ──')
        display(pd.DataFrame(rows))
    else:
        print(f'{label}: no alternatives found.')


# Weekly DM tests (need to rerun 03_arima_baseline.ipynb to get these)
run_dm_tests({
    'ARIMA rolling':              '../data/processed/preds_arima_rol.csv',
    'ARIMAX rolling':             '../data/processed/preds_arimax_rol.csv',
    'ARIMAX+Reddit':              '../data/processed/preds_arimax_reddit_rol.csv',
    'ARIMAX+News':                '../data/processed/preds_arimax_news_rol.csv',
    'ARIMAX+Reddit+News':         '../data/processed/preds_arimax_both_rol.csv',
}, baseline_name='ARIMAX rolling', label='Weekly DM tests')

# Daily LSTM DM tests
run_dm_tests({
    'LSTM-EXOG':    '../data/processed/preds_lstm_lstm-exog.csv',
    'LSTM-REDDIT':  '../data/processed/preds_lstm_lstm-reddit.csv',
    'LSTM-NEWS':    '../data/processed/preds_lstm_lstm-news.csv',
    'LSTM-Y':       '../data/processed/preds_lstm_lstm-y.csv',
}, baseline_name='LSTM-EXOG', label='Daily LSTM DM tests')


## 6. Sentiment ablation — does retail sentiment improve forecasts?

In [ ]:
# Baseline model names (exact strings from metrics CSVs)
BASELINES = {
    'Weekly ARIMAX':  'ARIMAX rolling   (lagged)',
    'Daily ARIMAX':   'ARIMAX rolling   (lagged)',   # same name, different frequency
    'Daily LSTM':     'LSTM-EXOG',
}
# These sentiment variant names are written by the respective notebooks
SENTIMENT_VARIANTS = [
    # Weekly (need rerun of 03_arima_baseline.ipynb)
    ('Weekly ARIMAX', 'ARIMAX+Reddit rolling   (lagged)'),
    ('Weekly ARIMAX', 'ARIMAX+News rolling   (lagged)'),
    ('Weekly ARIMAX', 'ARIMAX+Reddit+News rolling   (lagged)'),
    # Daily ARIMA (from metrics_arima_daily.csv)
    ('Daily ARIMA',  'ARIMAX+Reddit rolling   (daily)'),
    ('Daily ARIMA',  'ARIMAX+News rolling     (daily)'),
    ('Daily ARIMA',  'ARIMAX+Reddit+News rolling (daily)'),
    # Daily LSTM (from metrics_lstm.csv)
    ('Daily LSTM',   'LSTM-REDDIT'),
    ('Daily LSTM',   'LSTM-NEWS'),
]

rows = []
for group, variant_name in SENTIMENT_VARIANTS:
    variant = all_metrics[all_metrics['model'] == variant_name]
    if variant.empty:
        continue
    # find baseline for this group
    if group.startswith('Weekly'):
        base = all_metrics[(all_metrics['model'] == BASELINES['Weekly ARIMAX']) &
                           (all_metrics['frequency'] == 'Weekly')]
    elif group == 'Daily ARIMA':
        base = all_metrics[(all_metrics['model'] == BASELINES['Daily ARIMAX']) &
                           (all_metrics['frequency'] == 'Daily') &
                           (all_metrics['family'] == 'ARIMA')]
    else:  # LSTM
        base = all_metrics[all_metrics['model'] == BASELINES['Daily LSTM']]

    if base.empty:
        continue
    rows.append({
        'Group': group,
        'Model': variant_name,
        'DA':    variant['da'].values[0],
        'WDA':   variant['wda'].values[0],
        'ΔDA':   variant['da'].values[0]  - base['da'].values[0],
        'ΔWDA':  variant['wda'].values[0] - base['wda'].values[0],
    })

if rows:
    abl = pd.DataFrame(rows)
    display(abl.round(4))
    print('\nPositive ΔDA / ΔWDA = sentiment adds value over baseline.')
    print('Compare with DM test p-values above for statistical significance.')
else:
    print('No sentiment variant rows found. Re-run 03_arima_baseline.ipynb to add weekly variants.')


## 7. Silver Squeeze episode — retail hypothesis test

Notebook `05b_lstm_squeeze.ipynb` retrains on pre-2021 data (2015–2020) and
evaluates on the 2021 squeeze year, splitting into four sub-periods.

**Key question:** does LSTM-REDDIT outperform LSTM-EXOG specifically during the
squeeze peak (Jan 28 – Feb 5, 2021)?


In [ ]:
squeeze_path = '../data/processed/metrics_lstm_squeeze.csv'
if os.path.exists(squeeze_path):
    sq = pd.read_csv(squeeze_path)
    pivot = sq.pivot_table(index='model', columns='period', values='da')
    col_order = [c for c in [
        'Pre-peak   (Jan 01–27)',
        'Squeeze peak (Jan 28–Feb 05)',
        'Post-squeeze (Feb 06–Jun 30)',
        'Rest of 2021 (Jul–Dec)',
        'Full 2021'
    ] if c in pivot.columns]
    if col_order:
        pivot = pivot[col_order]

    print('Directional Accuracy — 2021 Squeeze Test (trained on 2015–2020):')
    display(pivot.round(3))

    if not pivot.empty and len(col_order) > 0:
        fig, ax = plt.subplots(figsize=(11, 3.5))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                    center=0.5, vmin=0.3, vmax=0.75, ax=ax,
                    linewidths=0.5, cbar_kws={'label': 'DA'})
        ax.set_title('DA by Model and Period — 2021 Silver Squeeze\n'
                     '(pre-squeeze retrain; orange = squeeze peak)',
                     fontweight='bold')
        # Highlight squeeze peak column in orange
        if 'Squeeze peak (Jan 28–Feb 05)' in col_order:
            cidx = col_order.index('Squeeze peak (Jan 28–Feb 05)')
            ax.add_patch(plt.Rectangle((cidx, 0), 1, len(pivot),
                                        fill=False, edgecolor='orange', lw=3))
        plt.tight_layout()
        plt.savefig('../data/processed/da_squeeze_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('metrics_lstm_squeeze.csv not found.')
    print('Run 05b_lstm_squeeze.ipynb to generate squeeze results.')


## 8. Export LaTeX tables

In [ ]:
# Weekly table
weekly_latex = (weekly[['model', 'rmse', 'mae', 'da', 'wda']]
                .sort_values('da', ascending=False)
                .rename(columns={'model': 'Model', 'rmse': 'RMSE', 'mae': 'MAE'}))
tex_w = weekly_latex.to_latex(
    index=False, float_format='%.4f',
    caption='Out-of-sample weekly forecast accuracy (test set: 2023--2024). '
            'DA = Directional Accuracy; WDA = Weighted DA.',
    label='tab:weekly_results')
with open('../data/processed/results_weekly.tex', 'w') as f:
    f.write(tex_w)
print('Saved results_weekly.tex')
print(tex_w)

# Daily table — LSTM only (comparable to each other)
lstm_only = all_metrics[all_metrics['family'] == 'LSTM'][['model', 'rmse', 'mae', 'da', 'wda']]
tex_d = lstm_only.sort_values('da', ascending=False).rename(
    columns={'model': 'Model', 'rmse': 'RMSE', 'mae': 'MAE'}).to_latex(
    index=False, float_format='%.4f',
    caption='LSTM variant performance (daily, test set: 2023--2024).',
    label='tab:lstm_results')
with open('../data/processed/results_lstm.tex', 'w') as f:
    f.write(tex_d)
print('Saved results_lstm.tex')
print(tex_d)
